# ComfyUI no Colab — instalação sob demanda por workflow

O *código* do ComfyUI fica no disco local do Colab (`/content`, rápido).
Modelos, outputs, inputs, workflows e o cache de custom nodes ficam no Drive.

**Só o ComfyUI-Manager é instalado sempre.** Todo o resto depende de quais
workflows você marcar na Célula 3 — o notebook lê o JSON de cada workflow,
descobre os `class_type` usados e instala apenas os pacotes necessários.

Ordem: **1 → 2 → 3 → 4 → (5 opcional) → 6**


In [ ]:
#@title 1. Montar Drive + instalar ComfyUI (local) { display-mode: "form" }
import os, subprocess, pathlib, json
from google.colab import drive

DRIVE_ROOT = '/content/drive'
DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'  #@param {type:"string"}
COMFY      = '/content/ComfyUI'

if not os.path.ismount(DRIVE_ROOT):
    drive.mount(DRIVE_ROOT)

def sh(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

if not os.path.exists(COMFY):
    sh(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}')
else:
    sh('git pull', cwd=COMFY, check=False)

MODEL_DIRS = ['checkpoints','loras','vae','clip','clip_vision','controlnet',
              'upscale_models','embeddings','unet','diffusion_models','ipadapter',
              'animatediff_models','animatediff_motion_lora','sams','style_models',
              'text_encoders','skintoken','trellis2','birefnet']
for d in MODEL_DIRS + ['ultralytics/bbox','ultralytics/segm']:
    pathlib.Path(f'{DRIVE_DATA}/models/{d}').mkdir(parents=True, exist_ok=True)
for d in ['output','input','user','workflows','node_cache']:
    pathlib.Path(f'{DRIVE_DATA}/{d}').mkdir(parents=True, exist_ok=True)

y = 'drive:\n  base_path: ' + DRIVE_DATA + '/models/\n'
y += ''.join(f'  {d}: {d}\n' for d in MODEL_DIRS)
y += '  ultralytics_bbox: ultralytics/bbox\n  ultralytics_segm: ultralytics/segm\n'
open(f'{COMFY}/extra_model_paths.yaml','w').write(y)

sh('pip install -q -r requirements.txt', cwd=COMFY)

# Manager: sempre. E o cache de nodes do Drive volta para o disco local.
CN = f'{COMFY}/custom_nodes'
CACHE = f'{DRIVE_DATA}/node_cache'
if not os.path.exists(f'{CN}/ComfyUI-Manager'):
    sh(f'git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager "{CN}/ComfyUI-Manager"')
    sh(f'pip install -q -r "{CN}/ComfyUI-Manager/requirements.txt"', check=False)

print('\n✅ Célula 1 OK. Coloque seus workflows .json em:', f'{DRIVE_DATA}/workflows')


In [ ]:
#@title 2. Registry de nodes (mapa class_type -> repositorio) { display-mode: "form" }
#@markdown Baixa o registry + os workflows direto do seu repo no GitHub.
import json, os, subprocess, urllib.request

DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'
REPO = 'https://github.com/BloomRX/ComfyUI_Collab'
BRANCH = 'arena/01a05a82-comfyui-collab'
CKOUT = '/content/ComfyUI_Collab'

if os.path.exists(CKOUT):
    subprocess.run('git pull', shell=True, cwd=CKOUT, check=False)
else:
    subprocess.run(f'git clone --depth 1 -b {BRANCH} {REPO} {CKOUT}', shell=True, check=True)

LOCAL = f'{DRIVE_DATA}/node_registry.json'
REGISTRY = json.load(open(LOCAL if os.path.exists(LOCAL) else f'{CKOUT}/config/node_registry.json'))

PACKS      = REGISTRY['packs']
CLASS_MAP  = REGISTRY['class_map']
NATIVE_IGNORE = set(REGISTRY.get('native_ignore', []))
WF_DIRS = [f'{CKOUT}/Workflows', f'{DRIVE_DATA}/workflows']
print(f'OK: {len(PACKS)} pacotes, {len(CLASS_MAP)} nodes mapeados.')


In [ ]:
#@title 3. Escolher os workflows desta sessao { display-mode: "form" }
#@markdown Marque so o que vai usar agora. Um workflow por sessao e o ideal.
import json, os, glob, re
import ipywidgets as W
from IPython.display import display

uuidpat = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-')
files = []
for d in WF_DIRS:
    files += sorted(glob.glob(f'{d}/**/*.json', recursive=True))

def classes_of(path):
    try: d = json.load(open(path, encoding='utf-8'))
    except Exception: return set()
    out = set()
    if isinstance(d, dict) and 'nodes' in d:
        for n in d['nodes']:
            if n.get('type'): out.add(n['type'])
    elif isinstance(d, dict):
        for n in d.values():
            if isinstance(n, dict) and n.get('class_type'): out.add(n['class_type'])
    return out

BOXES = []
if not files:
    print('Nenhum workflow encontrado em', WF_DIRS)
else:
    for f in files:
        cls  = classes_of(f)
        need = sorted({CLASS_MAP[c] for c in cls if c in CLASS_MAP})
        unk  = sorted(c for c in cls if c not in CLASS_MAP
                      and c not in NATIVE_IGNORE and not uuidpat.match(c))
        name = os.path.basename(f)
        label = f'{name}  ->  {", ".join(need) if need else "so nodes nativos"}'
        if unk: label += f'   [!] desconhecidos: {", ".join(unk[:4])}'
        b = W.Checkbox(value=False, description=label, indent=False,
                       layout=W.Layout(width='100%'))
        b._path, b._need = f, need
        BOXES.append(b)
    display(W.VBox([W.HTML('<b>Marque os workflows desta sessao:</b>')] + BOXES))
    print('\nDepois de marcar, rode a Celula 4.')


In [ ]:
#@title 4. Instalar os custom nodes dos workflows marcados { display-mode: "form" }
import os, subprocess

COMFY='/content/ComfyUI'; CN=f'{COMFY}/custom_nodes'
DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'; CACHE=f'{DRIVE_DATA}/node_cache'

selected = [b for b in BOXES if b.value]
needed = sorted({p for b in selected for p in b._need})
print('Workflows:', [os.path.basename(b._path) for b in selected] or '(nenhum)')
print('Pacotes necessários:', needed or '(nenhum — só nodes nativos)')

def sh(c, **k): print(f'$ {c}'); return subprocess.run(c, shell=True, check=False, **k)

for pack in needed:
    url = PACKS.get(pack)
    if not url:
        print(f'⚠️  {pack} não está em PACKS — instale pelo Manager.'); continue
    dst = f'{CN}/{pack}'
    if os.path.exists(dst + '.disabled') and not os.path.exists(dst):
        os.rename(dst + '.disabled', dst); print(f'reativado {pack}')
    if not os.path.exists(dst):
        sh(f'git clone --depth 1 {url} "{dst}"')
    req = f'{dst}/requirements.txt'
    if os.path.exists(req): sh(f'pip install -q -r "{req}"')
    # instaladores proprios (TRELLIS2 compila extensoes CUDA; SkinToken idem)
    if os.path.exists(f'{dst}/install.py'):
        print(f'--- {pack}: install.py (pode demorar bastante)')
        sh(f'python install.py', cwd=dst)

# Desativa o que não foi pedido nesta sessão (Manager nunca é desativado)
for d in sorted(os.listdir(CN)):
    p = f'{CN}/{d}'
    if not os.path.isdir(p) or d in ('__pycache__',): continue
    if d == 'ComfyUI-Manager' or d in needed: continue
    os.rename(p, p + '.disabled'); print(f'⏸️  {d} desativado nesta sessão')

print('\n✅ Ativos:', [d for d in sorted(os.listdir(CN))
                        if os.path.isdir(f'{CN}/{d}') and not d.endswith('.disabled')])


In [ ]:
#@title 5. (Opcional) Baixar modelo para o Drive { display-mode: "form" }
URL = ''  #@param {type:"string"}
PASTA = 'checkpoints'  #@param ["checkpoints","loras","vae","controlnet","upscale_models","unet","ipadapter","animatediff_models","clip_vision","sams","ultralytics/bbox"]
HF_TOKEN = ''  #@param {type:"string"}
import subprocess
DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'
if URL:
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}" ' if HF_TOKEN else ''
    subprocess.run(f'wget -c {hdr}--content-disposition "{URL}" -P "{DRIVE_DATA}/models/{PASTA}"',
                   shell=True, check=False)
else:
    print('Cole uma URL e rode de novo.')


In [ ]:
#@title 6. Ligar o ComfyUI { display-mode: "form" }
TUNEL = 'cloudflared'  #@param ["cloudflared","ngrok"]
VRAM  = 'normalvram'   #@param ["normalvram","highvram","lowvram"]

import subprocess, threading, re, time, os
COMFY='/content/ComfyUI'; DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'; PORT=8188

if TUNEL == 'cloudflared':
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('wget -q -O /usr/local/bin/cloudflared '
          'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
          '&& chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    def tunnel():
        p = subprocess.Popen(['cloudflared','tunnel','--url',f'http://127.0.0.1:{PORT}'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            m = re.search(r'https://[-\w.]+\.trycloudflare\.com', line)
            if m: print('\n🚀 LINK DE ACESSO:', m.group(0), '\n')
    threading.Thread(target=tunnel, daemon=True).start(); time.sleep(3)
else:
    import getpass
    subprocess.run('pip install -q pyngrok', shell=True, check=True)
    from pyngrok import ngrok
    ngrok.kill(); ngrok.set_auth_token(getpass.getpass('Ngrok authtoken: '))
    print('\n🚀 LINK DE ACESSO:', ngrok.connect(PORT,'http').public_url, '\n')

!cd {COMFY} && python main.py \
  --listen 127.0.0.1 --port {PORT} --{VRAM} \
  --output-directory "{DRIVE_DATA}/output" \
  --input-directory "{DRIVE_DATA}/input" \
  --user-directory "{DRIVE_DATA}/user" \
  --preview-method auto --disable-auto-launch


## Sobre VRAM e abrir tudo junto

Os 6 GB são o pico de **um** workflow rodando. O problema de abrir os três juntos não é
o pico — é que o ComfyUI mantém em VRAM o último modelo carregado de cada execução, e
aba aberta com workflow grande também custa RAM de CPU. Misturar SDXL + AnimateDiff +
3D na mesma sessão faz o Colab começar a fazer swap e, no T4 (16 GB), OOM.

Por isso a Célula 3: uma sessão = um propósito.

- **Um workflow por sessão** é o ideal.
- Precisa alternar? Use **Free model and node cache** no menu do ComfyUI antes de trocar.
- `lowvram` na Célula 6 se estourar; `highvram` só em A100.

## Como funciona a seleção

1. Salve os workflows em `ComfyUI_Data/workflows/` (pode usar subpastas: `splash/`, `3d/`, `anim/`).
2. Célula 3 lê o JSON de cada um, extrai os `class_type` e cruza com o `class_map` do registry.
3. Célula 4 clona só os pacotes daqueles workflows, e renomeia o resto para `.disabled`
   — não apaga nada, e reativar é só marcar o workflow de novo.
4. Os repos ficam em cache em `ComfyUI_Data/node_cache/`, então a segunda sessão não baixa nada.

## Quando aparecer um node desconhecido

Se um workflow usa um node que não está no `class_map`, ele não é instalado e o ComfyUI
mostra "missing node". Aí: **Manager → Install Missing Custom Nodes**, veja o nome do pacote,
e adicione o par `"NomeDoClassType": "NomeDoPacote"` em `config/node_registry.json`
(mais o repo em `packs`, se for novo). Na próxima sessão ele entra sozinho.

**Me manda os três workflows** que eu preencho o registry com os nodes reais deles e testo o parser.
